In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime
import random
import json
import pandas as pd

StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 3, Finished, Available, Finished)

In [2]:
!pip install azure-eventhub
import json
from azure.eventhub import EventHubProducerClient, EventData
from datetime import datetime

StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 4, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 23.5 MB/s eta 0:00:00


# Foot traffic Data

In [3]:
def generate_iot_metadata(dt):
    formatted_time = dt.strftime('%Y-%m-%dT%H:%M:%S.%f') + "000Z"
    metadata = {
        "MessageId": None,
        "CorrelationId": None,
        "ConnectionDeviceId": "foottraffic-device",
        "ConnectionDeviceGenerationId": "637769785707914893",
        "EnqueuedTime": formatted_time
    }
    return json.dumps(metadata)

store_ids = [f"S{str(i).zfill(3)}" for i in range(1, 13)]
aisles = ["Jackets & Coats", "Wine", "Solar Panels", "Paint", "Snacks", "Electronics", "Fleece Pullover"]

now = datetime.utcnow()
data_store_level = []
data_aisle_level = []

for store in store_ids:
    before_traffic = random.randint(10, 30)
    after_traffic = before_traffic + random.randint(5, 20)
    iot_hub = generate_iot_metadata(now)
    store_record = {
        "RecordedOn": now,
        "before_traffic": before_traffic,
        "after_traffic": after_traffic,
        "IoTHub": iot_hub,
        "storeid": store
    }
    data_store_level.append(store_record)

    aisle_weights = {
        "Jackets & Coats": 0.90,
        "Wine": 0.35,
        "Solar Panels": 0.30,
        "Paint": 0.15,
        "Snacks": 0.10,
        "Electronics": 0.10,
        "Fleece Pullover": 0.03  # low foot traffic for fleece pullover
    }
    total_weight = sum(aisle_weights.values())
    normalized_weights = {k: v / total_weight for k, v in aisle_weights.items()}

    aisle_after_counts = {}
    total_allocated = 0
    for aisle in aisles[:-1]:
        count = int(after_traffic * normalized_weights[aisle])
        aisle_after_counts[aisle] = count
        total_allocated += count
    last_aisle = aisles[-1]
    aisle_after_counts[last_aisle] = after_traffic - total_allocated

    for aisle in aisles:
        if aisle == "Fleece Pullover":
            aisle_before = max(1, int(before_traffic * (normalized_weights[aisle]*random.uniform(0.3, 0.7))))
        else:
            aisle_before = max(1, int(before_traffic * (normalized_weights[aisle]*random.uniform(0.7, 1.0))))
        aisle_after = aisle_after_counts[aisle]
        iot_hub_aisle = generate_iot_metadata(now)
        aisle_record = {
            "RecordedOn": now,
            "before_traffic": aisle_before,
            "after_traffic": aisle_after,
            "IoTHub": iot_hub_aisle,
            "storeid": store,
            "aisle_name": aisle
        }
        data_aisle_level.append(aisle_record)

# Example: print first 10 aisle-level records
print(data_aisle_level[:10])


StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 5, Finished, Available, Finished)

[{'RecordedOn': datetime.datetime(2025, 10, 8, 9, 34, 16, 488980), 'before_traffic': 6, 'after_traffic': 14, 'IoTHub': '{"MessageId": null, "CorrelationId": null, "ConnectionDeviceId": "foottraffic-device", "ConnectionDeviceGenerationId": "637769785707914893", "EnqueuedTime": "2025-10-08T09:34:16.488980000Z"}', 'storeid': 'S001', 'aisle_name': 'Jackets & Coats'}, {'RecordedOn': datetime.datetime(2025, 10, 8, 9, 34, 16, 488980), 'before_traffic': 3, 'after_traffic': 5, 'IoTHub': '{"MessageId": null, "CorrelationId": null, "ConnectionDeviceId": "foottraffic-device", "ConnectionDeviceGenerationId": "637769785707914893", "EnqueuedTime": "2025-10-08T09:34:16.488980000Z"}', 'storeid': 'S001', 'aisle_name': 'Wine'}, {'RecordedOn': datetime.datetime(2025, 10, 8, 9, 34, 16, 488980), 'before_traffic': 3, 'after_traffic': 4, 'IoTHub': '{"MessageId": null, "CorrelationId": null, "ConnectionDeviceId": "foottraffic-device", "ConnectionDeviceGenerationId": "637769785707914893", "EnqueuedTime": "2025-

In [ ]:
# Connection details - replace each one with the actual connection string of each Event Hub
# Paste Ingest_Aisle_Level_Foot_Traffic_Data eventstream - Event hub name,Connection string-primary key
CONNECTION_STR_AISLE = "#Connection string-primary key#"
EVENT_HUB_NAME_AISLE = "#Event hub name#"

# Paste Ingest_Store_Level_Foot_Traffic_Data eventstream - Event hub name,Connection string-primary key
CONNECTION_STR_STORE = "#Connection string-primary key#"
EVENT_HUB_NAME_STORE = "#Event hub name#"


# Initialize Event Hub producers with their respective connection strings
producer_store = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR_STORE, eventhub_name=EVENT_HUB_NAME_STORE)
producer_aisle = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR_AISLE, eventhub_name=EVENT_HUB_NAME_AISLE)

def send_batch(producer, records):
    batch = producer.create_batch()
    for record in records:
        json_payload = json.dumps(record, default=str)
        try:
            batch.add(EventData(json_payload))
        except ValueError:
            producer.send_batch(batch)
            batch = producer.create_batch()
            batch.add(EventData(json_payload))
    if len(batch) > 0:
        producer.send_batch(batch)

# Send store-level foot traffic data
send_batch(producer_store, data_store_level)
producer_store.close()

# Send aisle-level foot traffic data
send_batch(producer_aisle, data_aisle_level)
producer_aisle.close()


print("✅ Store-level and aisle-level foot traffic data sent to their respective Eventstreams.")


StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 6, Finished, Available, Finished)

✅ Store-level and aisle-level foot traffic data sent to their respective Eventstreams.


# Product inventory

In [6]:
from datetime import datetime, timedelta

now = datetime.now()

product_master = [
    ("WINE001", "Crimson Cabernet", "Beverages - Wine"),
    ("WINE002", "Golden Chardonnay", "Beverages - Wine"),
    ("WINE003", "Velvet Merlot", "Beverages - Wine"),
    ("WINE004", "Crystal Riesling", "Beverages - Wine"),
    ("WINE005", "Midnight Zinfandel", "Beverages - Wine"),
    ("SOLAR001", "SunPower Panel X", "Solar"),
    ("SOLAR002", "EcoLite Solar Module", "Solar"),
    ("APP001", "Winter Jacket", "Apparel"),
    ("APP002", "Raincoat", "Apparel"),
    ("APP003", "Fleece Pullover", "Apparel")
]

stores = ["Los Angeles", "Boston", "Seattle", "Ontario", "San Francisco", "Mountain View", "Chicago", "New York", "Atlanta", "Denver", "Dallas", "Miami"]

def generate_inventory():
    records = []
    ts = now.strftime("%Y-%m-%d %H:%M:%S")
    last_month_date = (now - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")
    for store in stores:
        for pid, pname, pcat in product_master:
            if "Wine" in pcat:
                stock_qty = random.choices([0, 1, 5, 10], weights=[0.45, 0.15, 0.3, 0.1])[0]
            else:
                stock_qty = random.randint(10, 100)
            restock_time = last_month_date if pname == "Fleece Pullover" else ts
            records.append({
                "StoreLocation": store,
                "ProductID": pid,
                "ProductName": pname,
                "Category": pcat,
                "InventorySKU": stock_qty,
                "LastRestockDateTime": restock_time
            })
    return pd.DataFrame(records)

df_inventory = generate_inventory()
print("Inventory Sample")
print(df_inventory.head())


StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 7, Finished, Available, Finished)

Inventory Sample
  StoreLocation ProductID         ProductName          Category  InventorySKU  \
0   Los Angeles   WINE001    Crimson Cabernet  Beverages - Wine             0   
1   Los Angeles   WINE002   Golden Chardonnay  Beverages - Wine             0   
2   Los Angeles   WINE003       Velvet Merlot  Beverages - Wine             5   
3   Los Angeles   WINE004    Crystal Riesling  Beverages - Wine             5   
4   Los Angeles   WINE005  Midnight Zinfandel  Beverages - Wine             0   

   LastRestockDateTime  
0  2025-10-08 09:34:18  
1  2025-10-08 09:34:18  
2  2025-10-08 09:34:18  
3  2025-10-08 09:34:18  
4  2025-10-08 09:34:18  


In [ ]:
# Replace with your actual Event Hub connection string and hub name for inventory
# Paste Ingest_Products_Inventory_Data eventstream - Event hub name,Connection string-primary key

CONNECTION_STR_INVENTORY = "#Connection string-primary key#"
EVENT_HUB_NAME_INVENTORY = "#Event hub name#"

# Initialize Event Hub producer for inventory
producer_inventory = EventHubProducerClient.from_connection_string(conn_str=CONNECTION_STR_INVENTORY, eventhub_name=EVENT_HUB_NAME_INVENTORY)

def send_inventory_batch(producer, records):
    batch = producer.create_batch()
    for record in records:
        json_payload = json.dumps(record, default=str)
        try:
            batch.add(EventData(json_payload))
        except ValueError:
            # Batch full, send current batch and start a new one
            producer.send_batch(batch)
            batch = producer.create_batch()
            batch.add(EventData(json_payload))
    if len(batch) > 0:
        producer.send_batch(batch)

# Assuming df_inventory is a Pandas DataFrame or a list of dicts
# If using Spark, convert to pandas first: df_inventory.toPandas()
inventory_records = df_inventory.to_dict(orient='records')

# Send inventory records
send_inventory_batch(producer_inventory, inventory_records)
producer_inventory.close()

print("✅ Inventory data sent to Eventstream.")


StatementMeta(, 08901f3d-d8fc-4699-adba-4be40a1dd510, 8, Finished, Available, Finished)

✅ Inventory data sent to Eventstream.
